# Part 2: clean CourtListener time entries + build hand-label sample

**Input:** `courtlistener_scrape/data/raw/time_entries_all.csv.gz` (from team Drive; gitignored)

**Outputs** (in `part_2_clean_data/data/`):
- `intermediate/time_entries_attorney_clean.csv.gz`: full cleaned pool (gitignored, rebuilt by this notebook)
- `final/label_sample.csv`: 240-row stratified sample to hand-label (committed)

Block billing is labeled as its **own column**, separate from narrative quality.

In [1]:
import re
from pathlib import Path
import pandas as pd

# Find the repo root (the folder with .git), so this runs no matter where Jupyter was launched
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())
RAW = ROOT / "courtlistener_scrape/data/raw/time_entries_all.csv.gz"
assert RAW.exists(), f"Missing {RAW}: download time_entries_all.csv.gz from the team Drive into courtlistener_scrape/data/raw/"

OUT = ROOT / "part_2_clean_data/data"
INTERMEDIATE = OUT / "intermediate"; INTERMEDIATE.mkdir(parents=True, exist_ok=True)
FINAL = OUT / "final"; FINAL.mkdir(parents=True, exist_ok=True)

SEED = 210
PER_CASE = 20          # rows sampled per case (cases with fewer clean rows give what they have)

cols = ["date", "timekeeper", "hours", "narrative", "high_confidence", "case_name",
        "firm", "entry_type", "task_codes", "document_id", "absolute_url"]
df = pd.read_csv(RAW, usecols=cols, low_memory=False)
print(f"raw rows: {len(df):,}")

raw rows: 505,530


## 1. Keep usable attorney entries

In [2]:
df = df[(df.entry_type == "attorney") & (df.high_confidence == True) & df.narrative.notna()].copy()
print(f"attorney, high-confidence, has narrative: {len(df):,}")

attorney, high-confidence, has narrative: 236,471


## 2. Clean narratives
Scrape artifacts found in the data (not attorney writing):
- **Matter-name prefixes**: `Wildfire Claims Matters - ...`, `Non-Bankruptcy Litigation - ...`
- **PG&E (Cravath) inline matter codes**: `WILD`, `NONB`, `REGS`, `INVS` ... dropped mid-sentence
- **Sears page header**: `ITEMIZED SERVICES - 73217.0004 - Chapter 11 Restructuring Date Timekeeper/ Index`
- **Leading task codes**: `P210 Reviewed SB 350`

In [3]:
PGE_CODES = ["WILD", "NONB", "REGS", "INVS", "COMM", "GOVR", "ADMN", "CRAV", "TRVL",
             "LIOU", "PERA", "OPRS", "FEEO"]
PGE_RE = re.compile(r"\s+(?:" + "|".join(PGE_CODES) + r")(?=\s)")
SEARS_HDR = re.compile(r"^ITEMIZED SERVICES\s*-\s*[\d.]+\s*-\s*.*?Timekeeper/\s*Index\s*", re.I)
PREFIX = re.compile(r"^[A-Z][A-Za-z&/,. ]{2,60}?\s+-\s+(?=[A-Z])")
TASK_CODE = re.compile(r"^[A-Z]{1,2}\d{2,4}\s+")
COLON_PREFIX = re.compile(r"^[A-Z][A-Za-z&/ ]{2,40}:\s+(?=[A-Z])")         # "Fee/Employment Applications: Review..."
ROW_NUM = re.compile(r"^(?:[A-Z]{1,3}\s+)?\d{1,2}\s+(?=[A-Z])")           # "1 CORRESPOND...", "F 1 REVIEW..."
PAGE_TAIL = re.compile(r"\s*(?:\b(?:Mon|Tue|Wed|Thu|Fri|Sat|Sun)\s+\d{6,}|MATTER NAME).*$")  # bled-in page footer
DOC_ID = re.compile(r"\(\d{6,}\)")                                          # "(57947831)"

def clean(text, case):
    t = SEARS_HDR.sub("", text)
    t = PREFIX.sub("", t)
    t = COLON_PREFIX.sub("", t)
    t = TASK_CODE.sub("", t)
    t = ROW_NUM.sub("", t)
    t = PAGE_TAIL.sub("", t)
    t = DOC_ID.sub("", t)
    if case == "PG&E Corp.":
        t = PGE_RE.sub("", t)
    return re.sub(r"\s+", " ", t).strip()

df["narrative_raw"] = df.narrative
df["narrative"] = [clean(t, c) for t, c in zip(df.narrative, df.case_name)]
changed = (df.narrative != df.narrative_raw)
print(f"narratives changed by cleaning: {changed.sum():,} ({changed.mean():.1%})")

narratives changed by cleaning: 69,104 (29.2%)


## 3. Drop rows that aren't time narratives
- **Expenses** that slipped in (taxis, meals, duplicating, CourtCall)
- **Truncated** narratives cut off mid-sentence by the PDF parser (end in "in", "the", "of"...)
- **Too short** (< 2 words)

In [4]:
EXPENSE = re.compile(r"\b(?:TAXI|UBER|MEALS?|MILES|DUPLICATING|COURTCALL|DISB|GRTRAN|LODGING|AIRFARE|PACER|OVERTIME)\b|PRINT\(S\)"
                     r"|(?i:\bmeals? working late\b|\bworking meals?\b|\btaxi\b|\bcarfare\b|\bgogo\b|\brepro\b|\bbw copies\b|\bphotocop)")
TRUNC = re.compile(r"\b(?:in|the|of|to|for|with|and|re|a|an|on|from|regarding)$", re.I)

df["drop_reason"] = None
df.loc[df.narrative.str.contains(EXPENSE), "drop_reason"] = "expense"
df.loc[df.drop_reason.isna() & df.narrative.str.contains(TRUNC), "drop_reason"] = "truncated"
df.loc[df.drop_reason.isna() & (df.narrative.str.split().str.len() < 2), "drop_reason"] = "too_short"
print(df.drop_reason.value_counts(dropna=False))
df = df[df.drop_reason.isna()].drop(columns="drop_reason")

drop_reason
None         231988
truncated      3825
expense         353
too_short       305
Name: count, dtype: int64


## 4. De-duplicate
Exact repeats within a case (e.g. `DOCUMENT REVIEW` x 2,000) would swamp the sample.
Short entries and ALL CAPS are **kept**: that's real attorney writing.

In [5]:
df["_key"] = df.narrative.str.lower().str.replace(r"[^a-z0-9 ]", "", regex=True)
before = len(df)
df["dup_count"] = df.groupby(["case_name", "_key"]).narrative.transform("size")
df = df.drop_duplicates(["case_name", "_key"]).drop(columns="_key")
print(f"removed {before - len(df):,} duplicates -> {len(df):,} unique entries")

df.to_csv(INTERMEDIATE / "time_entries_attorney_clean.csv.gz", index=False)
df.groupby("case_name").agg(entries=("narrative", "size"), firms=("firm", "nunique"))

removed 36,701 duplicates -> 195,287 unique entries


,entries,firms
case_name,,
BlockFi Inc.,7767,9
Boy Scouts of America,62,1
Celsius Network LLC,6946,10
Energy Future Holdings Corp.,44,5
FTX Trading Ltd.,24605,8
LTL Management LLC,5486,16
Lehman Brothers Holdings Inc.,2578,2
Motors Liquidation Company (General Motors),1024,6
PG&E Corp.,67063,45


## 5. Stratified sample for hand-labeling
Equal rows per case so no single case (PG&E is ~40% of entries) dominates.
Within each case, rows are spread across firms (round-robin), so one big firm doesn't dominate.

In [6]:
# round-robin across firms: 1st random row from each firm, then 2nd from each, ...
shuffled = df.sample(frac=1, random_state=SEED)
shuffled["_r"] = shuffled.groupby(["case_name", shuffled.firm.fillna("?")]).cumcount()
sample = (shuffled.sort_values("_r", kind="stable")
                  .groupby("case_name").head(PER_CASE)
                  .drop(columns="_r")
                  .sample(frac=1, random_state=SEED)    # shuffle so cases are mixed while labeling
                  .reset_index(drop=True))
sample.insert(0, "sample_id", [f"CL-{i:04d}" for i in range(1, len(sample) + 1)])

# Label columns to fill in (same quality labels as synthetic v3, minus BLOCK_BILLED)
sample["quality_label"] = ""   # GOOD / SPECIFIC / VAGUE / MISSING_SUBJECT / DENSE_SHORTHAND / CLERICAL / JUNK (parse error -> excluded)
sample["block_billed"] = ""    # Y / N  (separate flag)
sample["labeler"] = ""
sample["notes"] = ""

out_cols = ["sample_id", "quality_label", "block_billed", "narrative", "hours", "timekeeper", "date",
            "case_name", "firm", "labeler", "notes", "narrative_raw", "dup_count",
            "document_id", "absolute_url"]
sample[out_cols].to_csv(FINAL / "label_sample.csv", index=False)
print(f"wrote {len(sample)} rows -> {(FINAL / 'label_sample.csv').relative_to(ROOT)}")
sample.case_name.value_counts()

wrote 240 rows -> part_2_clean_data/data/final/label_sample.csv


case_name
Boy Scouts of America                          20
BlockFi Inc.                                   20
Sears Holdings Corp.                           20
Lehman Brothers Holdings Inc.                  20
Energy Future Holdings Corp.                   20
LTL Management LLC                             20
Voyager Digital Holdings Inc.                  20
PG&E Corp.                                     20
Purdue Pharma L.P.                             20
Celsius Network LLC                            20
FTX Trading Ltd.                               20
Motors Liquidation Company (General Motors)    20
Name: count, dtype: int64